In [ ]:
import polars as pl
import plotly.express as px
from plotly.subplots import make_subplots
import plotly.graph_objects as go
import colorir as cl
import numpy as np
from pathlib import Path
import analyses.calculate_adh as adh
from analyses import io
import analyses.phylogeny as phy
from tqdm import tqdm

In [ ]:
celldf = (
    io
        .read_celldfs("../runs/evo/", levels=["replica"])
        .with_columns(
            replica=pl.col("replica").cast(pl.UInt32)
        )
        .sort(["replica", "wtime"])
)
celldf

In [ ]:
wtimes = celldf["wtime"].sort().diff()
wtime = wtimes.filter(wtimes != 0).mean()
selectors = ["index", "ligands", "receptors"]
gammadfs = []
meangammadfs = []
groupped = (
    celldf
        .select(selectors + ["replica", "wtime"])
        .filter((pl.col("wtime") - wtime) % 2e6 == 0)  # Values only change between seasons of 2e6 steps
        .group_by(["replica", "wtime"])
)
for (replica, time), group in tqdm(groupped):
    sel_group = group.select(selectors)
    lig_recep = sel_group.join(sel_group, how="cross")
    gammas = adh.calculate_gamma(
        20, 
        adh.cell_contact_energy(
            *lig_recep.select(pl.exclude("index", "index_right")).to_numpy().T,
            20,
            32,
            8
        )
    )
    means = lig_recep.with_columns(gamma=gammas).group_by("index").agg(
        pl.col("gamma").mean()
    )
    meangammadfs.append(means.with_columns(replica=replica, wtime=time))
    summary = (
        pl.DataFrame(gammas)
            .describe()
            .with_columns(replica=replica, wtime=time)
            .pivot(on="statistic", values="column_0")
    ).with_columns(sample=np.expand_dims(np.random.choice(gammas, 100), 0))
    gammadfs.append(summary)
gammadf = pl.concat(gammadfs).sort("replica", "wtime")
gammadf
    

In [ ]:
fig = make_subplots(1, 2, column_widths=[0.85, 0.15], shared_yaxes=True, horizontal_spacing=0.01)
colors = cl.StackPalette.load("carnival").resize(gammadf["replica"].n_unique(), repeat=True)
for (replica,), group in gammadf.sort("wtime").group_by("replica", maintain_order=True):
    color = colors[7]
    line = go.Scatter(
        x=group["wtime"],
        y=group["mean"],
        # Lines or markers
        mode="lines",
        line_color=color,
        line_width=6,
        opacity=0.1,
        legendgroup=replica,
        name=replica,
        marker_size=8,
        # visible="legendonly"
    )
    fig.add_trace(line, row=1, col=1)
fig.add_trace(go.Histogram(
    y=gammadf.filter(wtime=pl.col("wtime").max())["mean"],
    nbinsy=12,
    marker_color="#999999"
), row=1, col=2)

fig.update_layout(
    template="plotly_white",
    xaxis_title="time",
    yaxis_title="mean gamma",
    width=700,
    height=400,
    bargap=0.1,
    showlegend=False
)
fig.update_xaxes(
    showgrid=False,
    showticklabels=False,
    col=2
)
fig.update_yaxes(
    showgrid=False,
    col=2
)
fig.update_xaxes(tickangle=0)
io.save_plot(fig, "../plots/adhesion_evolution")

In [ ]:
def make_ancmelt(ancdf):
    return (
        ancdf
            # "lineage" is a unique identifier for leaf and corresponding ancestry path (both extinct and non-extinct)
            # "og" is the cell id of each lineage at the root (wtime == 0)
            # "id" is the cell id of the lineage at that time in the simulation
            .with_columns(lineage=np.arange(ancdf.height), og=ancdf["100000"])
            .unpivot(index=["lineage", "og"], value_name="index", variable_name="wtime")
            .with_columns(pl.col("wtime").cast(int))
            .sort(["lineage", "wtime"])
    )

In [ ]:
ancmelts = []
for r in tqdm(range(32)):
    ancdf = phy.matrix(celldf.filter(replica=r), 2e6)
    melt = make_ancmelt(ancdf).with_columns(replica=r)
    ancmelts.append(melt)
ancmelt = pl.concat(ancmelts)
ancmelt

In [ ]:
meangammadf = pl.concat(meangammadfs).sort("replica", "wtime", "gamma")
meangammadf

In [ ]:
gammadesc = ancmelt.join(
    ancmelt.select("replica", "wtime", "lineage", "index"),
    left_on=["replica", "wtime", "lineage"],
    right_on=["replica", pl.col("wtime") - 2_000_000, "lineage"],
    suffix="_desc"
).unique(["replica", "wtime", "index", "index_desc"]).join(
    meangammadf,
    on=["replica", "wtime", "index"],
).sort("replica", "wtime", "index")
gammadesc

In [ ]:
bins = np.linspace(gammadesc["wtime"].min(), gammadesc["wtime"].max(), 17)
fitdf = gammadesc.with_columns(
    rel_gamma=pl.col("gamma").clip(lower_bound=0) / pl.col("gamma").clip(lower_bound=0).mean().over("replica", "wtime"),
).group_by(
    "replica",
    "index",
    "wtime"
).agg(
    rel_gamma=pl.col("rel_gamma").first(),
    rel_gamma_check=pl.col("rel_gamma").n_unique(),
    n_desc=pl.len() - (pl.col("index_desc").null_count() == 1)
).with_columns(
    time_bin=pl.col("wtime").cut(bins[1:-1], labels=bins[:-1].round(2).astype("str")).cast(str).cast(float)
).drop_nans().sort("replica", "wtime", "rel_gamma")
assert (fitdf["rel_gamma_check"] == 1).all()
fitdf

In [ ]:
px.histogram(fitdf, x="rel_gamma", y="n_desc", histfunc="avg", animation_frame="time_bin")

In [ ]:
from scipy.optimize import minimize_scalar


def neg_log_likelihood(a, W, t, N):
    # This was selected by AIC compared to p = t * a
    p = 1 / N + t * a
    # Guard against log(0) or log(negative)
    if np.any(p <= 0) or np.any(p >= 1):
        return np.inf
    # No need to include the combinatorial since it doesnt depend on a
    return -np.sum(W * np.log(p) + (N - W) * np.log(1 - p))


def estimate_a(W, t, N):
    # Keep all p_i in (0,1)
    lower_bound = -1 / (N * np.max(t))
    upper_bound = (1 - 1/N) / np.max(t)
    result = minimize_scalar(
        neg_log_likelihood,
        bounds=(lower_bound, upper_bound),
        method='bounded',
        args=(W, t, N)
    )
    return result.x


def se_a_fisher(a_hat, t, N):
    p = 1 / N + t * a_hat
    # We get I(a) by applying the chain rule to I(p) = N / (p * (1 - p))
    fisher_info = np.sum(N * t**2 / (p * (1 - p)))
    return 1 / np.sqrt(fisher_info)


gens = 5 # Width of the window in number of generations
period = gens * 2_000_000  
start = 100_000
offset = period // 2
As = []
for (replica, time), group in fitdf.group_by_dynamic(
    "wtime",
    every="2000000i",  # Can increase this to grab less data points for efficiency
    period=f"{period}i",
    offset=f"-{offset - start}i",
    group_by="replica",
    closed="both"
):
    t = np.array(group["rel_gamma"])
    W = np.array(group["n_desc"])
    a = estimate_a(W, t, 100)
    se = se_a_fisher(a, t, 100)
    As.append((replica, time + offset, a, se, group["wtime"][0], group["wtime"][-1]))
binomdf = pl.from_records(
    As, 
    schema=("replica", "wtime", "a", "se", "start", "end"), 
    orient="row"
).filter(pl.col("wtime") <= pl.col("end").max()).sort("replica", "wtime")
binomdf

In [ ]:
# This is just to help understand how the group_by_dyn op works
df = pl.DataFrame({"int": range(0, 100, 3)})
df.group_by_dynamic(
    "int",
    every="3i",
    period="7i",
    offset="0i",
    start_by="window"
).agg(pl.col("int").alias("a")).with_columns(pl.col("int"))

In [ ]:
def compare_models_multiple(datasets, N):
    aic1_total, aic2_total = 0, 0
    
    for W, t in datasets:
        def neg_log_likelihood(a):
            p = t * a
            # Guard against log(0) or log(negative)
            if np.any(p <= 0) or np.any(p >= 1):
                return np.inf
            # No need to include the combinatorial since it doesnt depend on a
            return -np.sum(W * np.log(p) + (N - W) * np.log(1 - p))

        def nll_m2(a):
            p = 1/N + t * a
            if np.any(p <= 0) or np.any(p >= 1):
                return np.inf
            return -np.sum(W * np.log(p) + (N - W) * np.log(1 - p))

        res1 = minimize_scalar(neg_log_likelihood, bounds=(0, 1/np.max(t)), method='bounded')
        res2 = minimize_scalar(nll_m2, bounds=(-(1/N)/np.max(t), (1 - 1/N)/np.max(t)), method='bounded')

        aic1_total += 2*1 + 2*res1.fun
        aic2_total += 2*1 + 2*res2.fun

    print(f"Model 1 (p = t*a):       total AIC = {aic1_total:.4f}")
    print(f"Model 2 (p = 1/N + t*a): total AIC = {aic2_total:.4f}")
    print(f"Preferred: {'Model 1' if aic1_total < aic2_total else 'Model 2'}")

ds = []
for (replica, time), group in fitdf.filter(pl.col("rel_gamma") != 0).group_by("replica", "time_bin"):
    ds.append((np.array(group["n_desc"]), np.array(group["rel_gamma"])))
compare_models_multiple(ds, 100)

In [ ]:
fig = make_subplots(2, 1, shared_xaxes=True)
runs = [0, 5, 4]
fig.add_traces(px.line(
    binomdf.filter(pl.col("replica").is_in(runs)),
    x="wtime",
    y="a",
    color="replica",
    # error_y="se"
).data, rows=1, cols=1)
fig.add_traces(px.line(
    gammadf.filter(pl.col("replica").is_in(runs)),
    x="wtime",
    y="mean",
    color="replica"
).data, rows=2, cols=1)
fig.update_traces(
    # visible="legendonly"
)
fig.update_layout(
    template="plotly_white",
    width=600,
    height=450,
    showlegend=False,
    yaxis_title="a"
)
fig.update_xaxes(
    title="time",
    row=2
)
fig.update_yaxes(
    title="gamma",
    row=2
)
io.save_plot(fig, "../plots/rel_gamma-fitness")

In [ ]:
multimesdf = gammadf.with_columns(
    gamma_level=(
        pl.when(pl.col("mean") > 10)
            .then(pl.lit("high"))
            .otherwise(
                pl.when(pl.col("mean") <= 0)
                    .then(pl.lit("neg"))
                    .otherwise(pl.lit("low"))
            )
    ).cast(pl.Enum(("neg", "low", "high")))
).join(
    binomdf, 
    on=["replica", "wtime"]
).filter(
    # Filter out runs that are not high adhesion at the end of the sim
    ((pl.col("gamma_level") == "high") & (pl.col("wtime") == pl.col("wtime").max())).any().over("replica")
).sort("replica", "wtime")
multimesdf

In [ ]:
boxcolors = cl.StackPalette.load("safe")
fig = px.box(
    # Ensure independent measurements (unlike the wolling window)
    multimesdf.filter((pl.col("wtime") - 1e5) % period == 0).sort("gamma_level"),
    y="a",
    x="gamma_level", 
    color_discrete_sequence=boxcolors,
    notched=True,
)
fig.update_traces(
    marker_line_width=1,
    marker_line_color="white",
    marker_color=boxcolors[0],
    marker_opacity=0.5,
    jitter=0.5
)
fig.update_layout(
    template="plotly_white",
    width=400,
    height=400,
    xaxis_title="mean gamma of population"
)
io.save_plot(fig, "../plots/a_coeff_box")

In [ ]:
muldf = multimesdf.with_columns(
    rle_id=pl.col("gamma_level").rle_id().over("replica"),
).group_by("replica", "rle_id").agg(
    rle=pl.len(),
    gamma_level=pl.first("gamma_level"),
    midtime=(pl.col("start").min() + pl.col("end").max()) / 2
).filter(
    rle=pl.col("rle").filter(pl.col("gamma_level") == "low").max().over("replica")
).join(
    binomdf.filter(a=pl.col("a").max().over("replica")),
    on=["replica"]
).sort("wtime")
muldf

In [ ]:
import statsmodels.api as sm

X = sm.add_constant(muldf["wtime"].to_numpy())
model = sm.OLS(muldf["midtime"].to_numpy(), X).fit()
predictions = model.get_prediction(X)
summary = predictions.summary_frame(alpha=0.05)
print(model.summary(alpha=0.05))

fig = px.scatter(
    muldf,
    x="wtime", 
    y="midtime",
    hover_name="replica",
)
fig.update_traces(
    zorder=3,
    marker_color=boxcolors[0]
)
fig.add_traces([
    # go.Scatter(
    #     x=pl.concat([muldf["wtime"], muldf["wtime"][::-1]]),
    #     y=np.concatenate([summary["mean_ci_lower"], summary["mean_ci_upper"][::-1]]),
    #     fill="toself",
    #     line_color="rgba(0, 0, 0, 0)",
    #     opacity=0.2
    # ),
    go.Scatter(
        x=muldf["wtime"],
        y=summary["mean"],
        line_color="black"
    )
])
fig.update_layout(
    template="plotly_white",
    width=400,
    height=350,
    xaxis_title="time of peak a",
    yaxis_title="time of transition to mul.",
    showlegend=False,
    title=f"r^2: {model.rsquared}"
)
io.save_plot(fig, "../plots/peak_a-transition")

In [ ]:
px.line(ancmelt, x="index", y="wtime", color="lineage").update_traces(visible="legendonly")

In [ ]:
descdf = ancmelt.group_by(["og", "wtime"]).agg(n_descendents=pl.col("index").drop_nulls().n_unique()).sort("og", "wtime")
descdf

In [ ]:
# First common ancestor of whole pop at wtime = 58e6
px.bar(
    descdf,
    x="wtime", 
    y="n_descendents", 
    color="og",
    barmode="stack"
).update_traces(
    marker_line_color="rgba(0, 0, 0, 0)"
).update_layout(
    bargap=0
)